# Paired Sobol sensitivity

Edit MODEL_DIR and OUT; relative paths resolve from the kernel working directory.
Supply eight final <stage>_c.xml models and four original snapshots in
original_with_added_rxns/. Dependencies: COBRApy, NumPy, pandas, SALib and Gurobi with
a usable license. Inputs are not bundled.

Exchange bounds are matched to the final model within each stage. Only final-model
uptake-permitted exchanges vary, independently and uniformly from 0 to 10 model flux
units; secretion and internal bounds are preserved. Egg, L1, L2 and L3 have separate
original snapshots. Other stages are unchanged same-file controls with reused response
vectors. No cross-stage medium matching is imposed.

SALib estimates first-order S1 and total-order ST, with 500 bootstrap resamples and
95% normal-approximation half-widths. Paired fixed-minus-original intervals use
identical resample rows and 2.5th/97.5th percentiles. Indices are dimensionless;
fluxes retain model units, without a division-rate conversion. Negative or above-one
estimates are retained.

Nested base sizes are 128, 256, 512, 1024 and 2048. As in the source, the first solve
batch reaches 512 before checking the first three levels. Both versions must have
maximum S1/ST change ≤0.05 and maximum 95% half-width ≤0.10. Stop at 2048 if
unresolved. Nonoptimal responses are not discarded or replaced with zero; missing or
numerically constant outputs give undefined indices. Distribution summaries use
independent A/B rows only. Method: Nobile et al. (2021),
doi:10.1186/s12859-021-04002-0. No calculations were executed for this copy.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import cobra
from SALib.sample import sobol as sampler
from SALib.analyze import sobol as analyzer

MODEL_DIR = Path("inputs/models")
OUT = Path("results/sobol_paired_results")
OUT.mkdir(parents=True, exist_ok=True)
STAGES = ["egg", "L1", "L2", "L3", "L4f", "L4m", "Am", "Af"]
BACKUPS = {
    "egg": "egg_c_pre_MAR04670_07871.xml",
    "L1": "L1_c_pre_gapfill.xml",
    "L2": "L2_c_pre_gapfill.xml",
    "L3": "L3_c_pre_step1_boundsonly.xml",
}
SEED = 1729
BOOTSTRAP = 500
LEVELS = [128, 256, 512, 1024, 2048]
SOLVER_TOL = 1e-9


def sobol_indices(problem, responses):
    """Estimate complete-design indices; missing/constant outputs are undefined."""
    if not np.isfinite(responses).all():
        return None, "nonoptimal_output_present"
    if np.ptp(responses) <= 1e-10:
        return None, "constant_or_numerically_constant_output"
    return (
        analyzer.analyze(
            problem,
            responses,
            calc_second_order=False,
            num_resamples=BOOTSTRAP,
            conf_level=0.95,
            keep_resamples=True,
            seed=SEED,
        ),
        "estimated",
    )


paired_tables = []
model_rows = []
condition_rows = []
convergence_tables = []


In [ ]:
for stage in STAGES:
    models = {}
    for version in ["original", "fixed"]:
        path = (
            MODEL_DIR / "original_with_added_rxns" / BACKUPS[stage]
            if version == "original" and stage in BACKUPS
            else MODEL_DIR / f"{stage}_c.xml"
        )
        model = cobra.io.read_sbml_model(str(path))
        model.solver = "gurobi"
        model.tolerance = SOLVER_TOL
        model.solver.configuration.timeout = 120
        for reaction in model.reactions:
            reaction.id = reaction.id.replace("_Human-GEM", "")
        model.objective = model.reactions.get_by_id("MAR04413")
        model.objective.direction = "max"
        models[version] = model
    common_bounds = {r.id: r.bounds for r in models["fixed"].exchanges}
    assert set(common_bounds) == {r.id for r in models["original"].exchanges}
    for reaction_id, bounds in sorted(common_bounds.items()):
        original = models["original"].reactions.get_by_id(reaction_id)
        condition_rows.append(
            dict(
                stage=stage,
                reaction_id=reaction_id,
                original_lower=original.lower_bound,
                original_upper=original.upper_bound,
                common_lower=bounds[0],
                common_upper=bounds[1],
            )
        )
        original.bounds = bounds
    parameters = []
    for reaction in models["fixed"].exchanges:
        assert len(reaction.metabolites) == 1
        coefficient = next(iter(reaction.metabolites.values()))
        assert abs(coefficient) == 1
        endpoint = "lower" if coefficient < 0 else "upper"
        capacity = -reaction.lower_bound if coefficient < 0 else reaction.upper_bound
        if capacity <= 0:
            continue
        assert (
            reaction.upper_bound >= 0
            if endpoint == "lower"
            else reaction.lower_bound <= 0
        )
        parameters.append((reaction.id, endpoint))
    parameters.sort()
    dimension = len(parameters)
    assert dimension > 0
    problem = dict(
        num_vars=dimension,
        names=[r for r, _ in parameters],
        bounds=[[0.0, 10.0]] * dimension,
    )
    design = sampler.sample(
        problem, LEVELS[-1], calc_second_order=False, scramble=True, seed=SEED
    )
    np.savez_compressed(
        OUT / f"{stage}_sample_design_maxN.npz",
        capacity=design,
        reaction_ids=np.array(problem["names"]),
    )
    exchanges = {
        version: [
            (model.reactions.get_by_id(rid), endpoint) for rid, endpoint in parameters
        ]
        for version, model in models.items()
    }
    responses = {version: [] for version in models}
    statuses = {version: [] for version in models}
    previous = {}
    for base_size in LEVELS[2:]:
        target_rows = base_size * (dimension + 2)
        for sample_index in range(len(responses["original"]), target_rows):
            for version, model in models.items():
                if version == "fixed" and stage not in BACKUPS:
                    value = responses["original"][-1]
                    status = statuses["original"][-1]
                else:
                    for (reaction, endpoint), capacity in zip(
                        exchanges[version], design[sample_index]
                    ):
                        if endpoint == "lower":
                            reaction.lower_bound = -float(capacity)
                        else:
                            reaction.upper_bound = float(capacity)
                    try:
                        value = float(model.slim_optimize(error_value=np.nan))
                        status = str(model.solver.status)
                    except Exception as error:
                        value, status = np.nan, "error:" + type(error).__name__
                responses[version].append(value if status == "optimal" else np.nan)
                statuses[version].append(status)
        checked_sizes = LEVELS[:3] if base_size == 512 else [base_size]
        for checked_size in checked_sizes:
            latest = {}
            precision_checks = []
            for version in models:
                values = np.array(responses[version][: checked_size * (dimension + 2)])
                indices, state = sobol_indices(problem, values)
                latest[version] = (indices, state)
                change, halfwidth = np.nan, np.nan
                if indices is not None:
                    halfwidth = max(np.max(indices[k + "_conf"]) for k in ["S1", "ST"])
                    if version in previous:
                        change = max(
                            np.max(abs(indices[k] - previous[version][k]))
                            for k in ["S1", "ST"]
                        )
                    previous[version] = indices
                reached = bool(
                    np.isfinite(change) and change <= 0.05 and halfwidth <= 0.10
                )
                precision_checks.append(reached)
                convergence_tables.append(
                    pd.DataFrame(
                        dict(
                            stage=stage,
                            version=version,
                            N=checked_size,
                            reaction_id=problem["names"],
                            state=state,
                            S1=indices["S1"] if indices is not None else np.nan,
                            ST=indices["ST"] if indices is not None else np.nan,
                            S1_conf_halfwidth=(
                                indices["S1_conf"] if indices is not None else np.nan
                            ),
                            ST_conf_halfwidth=(
                                indices["ST_conf"] if indices is not None else np.nan
                            ),
                            max_change=change,
                            max_conf_halfwidth=halfwidth,
                            precision_target_met=reached,
                        )
                    )
                )
        converged = all(precision_checks)
        if converged or any(indices is None for indices, _ in latest.values()):
            break
    paired = pd.DataFrame(
        dict(
            stage=stage,
            reaction_id=problem["names"],
            name=[
                models["fixed"].reactions.get_by_id(r).name for r in problem["names"]
            ],
            endpoint=[endpoint for _, endpoint in parameters],
            capacity_min=0.0,
            capacity_max=10.0,
            N=base_size,
            precision_target_met=converged,
            independent_original=stage in BACKUPS,
        )
    )
    for version, model in models.items():
        for reaction, _ in exchanges[version]:
            reaction.bounds = common_bounds[reaction.id]
        values = np.array(responses[version])
        status_array = np.array(statuses[version])
        np.savez_compressed(
            OUT / f"{stage}_{version}_responses.npz",
            biomass=values,
            status=status_array,
        )
        independent = np.concatenate(
            [values[:: dimension + 2], values[dimension + 1 :: dimension + 2]]
        )
        finite = independent[np.isfinite(independent)]
        model_rows.append(
            dict(
                stage=stage,
                version=version,
                D=dimension,
                N=base_size,
                independent_original=stage in BACKUPS,
                evaluations=len(values),
                optimal=int((status_array == "optimal").sum()),
                infeasible=int((status_array == "infeasible").sum()),
                other_status=int(
                    (~np.isin(status_array, ["optimal", "infeasible"])).sum()
                ),
                biomass_index_state=latest[version][1],
                precision_target_met=converged,
                mean_biomass=float(finite.mean()) if len(finite) else np.nan,
                std_biomass=float(finite.std(ddof=1)) if len(finite) > 1 else np.nan,
                min_biomass=float(finite.min()) if len(finite) else np.nan,
                max_biomass=float(finite.max()) if len(finite) else np.nan,
            )
        )
        indices, state = latest[version]
        paired[version + "_index_state"] = state
        for key in ["S1", "ST"]:
            paired[key + "_" + version] = (
                indices[key] if indices is not None else np.nan
            )
            paired[key + "_" + version + "_CI95_halfwidth"] = (
                indices[key + "_conf"] if indices is not None else np.nan
            )
        paired["ST_minus_S1_" + version] = (
            paired["ST_" + version] - paired["S1_" + version]
        )
        if indices is not None:
            np.savez_compressed(
                OUT / f"{stage}_{version}_bootstrap.npz",
                S1=indices["S1_conf_all"],
                ST=indices["ST_conf_all"],
            )
        if set(status_array).issubset({"optimal", "infeasible"}):
            feasibility, feasibility_state = sobol_indices(
                problem, (status_array == "optimal").astype(float)
            )
            pd.DataFrame(
                dict(
                    reaction_id=problem["names"],
                    state=feasibility_state,
                    S1=feasibility["S1"] if feasibility is not None else np.nan,
                    ST=feasibility["ST"] if feasibility is not None else np.nan,
                )
            ).to_csv(
                OUT / f"{stage}_{version}_feasibility_indices.tsv",
                sep="\t",
                index=False,
            )
    original_indices, final_indices = latest["original"][0], latest["fixed"][0]
    for key in ["S1", "ST"]:
        paired[key + "_difference_fixed_minus_original"] = (
            paired[key + "_fixed"] - paired[key + "_original"]
        )
        lower = upper = np.full(dimension, np.nan)
        if original_indices is not None and final_indices is not None:
            differences = (
                final_indices[key + "_conf_all"] - original_indices[key + "_conf_all"]
            )
            lower, upper = np.quantile(differences, [0.025, 0.975], axis=0)
        paired[key + "_difference_CI95_lower"] = lower
        paired[key + "_difference_CI95_upper"] = upper
    paired_tables.append(paired)


In [ ]:
pd.concat(paired_tables, ignore_index=True).to_csv(
    OUT / "sobol_original_fixed_comparison.tsv", sep="\t", index=False
)
pd.DataFrame(model_rows).to_csv(OUT / "model_summary.tsv", sep="\t", index=False)
pd.DataFrame(condition_rows).to_csv(
    OUT / "common_exchange_conditions.tsv", sep="\t", index=False
)
pd.concat(convergence_tables, ignore_index=True).to_csv(
    OUT / "convergence_indices_all.tsv", sep="\t", index=False
)
